# 奇異矩陣 (Singular Matrix)

本筆記本延續前幾章的行列式、反矩陣與秩，專門討論**奇異矩陣**：

1. **定義** — 什麼是奇異／非奇異矩陣
2. **等價條件** — $\det$、秩、可逆性、列／行線性相關
3. **幾何意義** — 體積塌縮為零
4. **齊次方程組** — $A\mathbf{x}=\mathbf{0}$ 的非平凡解
5. **非齊次方程組** — 奇異時無解或無限多解
6. **綜合練習**

In [1]:
import sympy as sp
from IPython.display import display, Math

def display_large(expr):
    """Display a sympy expression in LaTeX with large font size."""
    display(Math(r"\large{%s}" % sp.latex(expr)))


def display_huge(expr):
    """Display a sympy expression in LaTeX with huge font size."""
    display(Math(r"\huge{%s}" % sp.latex(expr)))

## 1. 什麼是奇異矩陣？

對**方陣** $A$（$n \times n$）而言：

| 名稱 | 英文 | 意義 |
|------|------|------|
| **奇異矩陣** | Singular | **不可逆**，沒有反矩陣 $A^{-1}$ |
| **非奇異矩陣** | Nonsingular | **可逆**，存在 $A^{-1}$ 使得 $A A^{-1} = I$ |

最常用的判定：

$$
A \text{ 奇異} \quad \Leftrightarrow \quad \det(A) = 0
$$

In [2]:
# Singular vs nonsingular examples
S = sp.Matrix([
    [1, 2],
    [2, 4],   # row 2 = 2 * row 1  =>  singular
])

N = sp.Matrix([
    [1, 2],
    [3, 5],   # rows independent  =>  nonsingular
])

print("S (singular candidate):")
display_huge(S)
print("det(S) =")
display_large(S.det())
print("Singular?", S.det() == 0)

print("\nN (nonsingular candidate):")
display_huge(N)
print("det(N) =")
display_large(N.det())
print("Singular?", N.det() == 0)

S (singular candidate):


<IPython.core.display.Math object>

det(S) =


<IPython.core.display.Math object>

Singular? True

N (nonsingular candidate):


<IPython.core.display.Math object>

det(N) =


<IPython.core.display.Math object>

Singular? False


## 2. 等價條件一覽

對 $n \times n$ 方陣 $A$，以下敘述**全部等價**：

1. $A$ 是**奇異**的（不可逆）
2. $\det(A) = 0$
3. $\operatorname{rank}(A) < n$（不是滿秩）
4. RREF 不是單位矩陣 $I$
5. 列向量**線性相關**（某一列是其他列的線性組合）
6. 行向量**線性相關**
7. 齊次方程 $A\mathbf{x} = \mathbf{0}$ 有**非零解**（零空間維度 $\ge 1$）

反過來：若 $\det(A) \neq 0$，則以上全部不成立，$A$ 為非奇異。

In [3]:
def diagnose(A, name="A"):
    """Print equivalent singularity checks for a square matrix."""
    print(f"=== {name} ===")
    display_huge(A)
    det_A = A.det()
    rref_A, pivots = A.rref()
    print("det =")
    display_large(det_A)
    print(f"rank = {A.rank()}  (n = {A.rows})")
    print("RREF =")
    display_huge(rref_A)
    print("Pivots:", list(pivots))
    print("Singular?", det_A == 0)
    print("Invertible?", det_A != 0)
    print()


diagnose(S, "S")
diagnose(N, "N")

=== S ===


<IPython.core.display.Math object>

det =


<IPython.core.display.Math object>

rank = 1  (n = 2)
RREF =


...

det =


<IPython.core.display.Math object>

rank = 2  (n = 2)
RREF =


<IPython.core.display.Math object>

Pivots: [0, 1]
Singular? False
Invertible? True



## 3. 3×3 奇異例子：列線性相關

若某一列是其他列的組合，行列式必為零。例如：

$$
A = \begin{pmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{pmatrix}
\quad\text{（注意：} R_3 = 2 R_2 - R_1 \text{）}
$$

In [4]:
A = sp.Matrix([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])

print("A =")
display_huge(A)

# Verify linear dependence: R3 = 2*R2 - R1
R1, R2, R3 = A.row(0), A.row(1), A.row(2)
print("2*R2 - R1 =")
display_large(2 * R2 - R1)
print("Equals R3?", 2 * R2 - R1 == R3)

print("det(A) =")
display_large(A.det())
print("rank(A) =", A.rank())

try:
    A.inv()
except Exception as e:
    print("A.inv() raises:", type(e).__name__, "—", e)

A =


<IPython.core.display.Math object>

2*R2 - R1 =


<IPython.core.display.Math object>

Equals R3? True
det(A) =


<IPython.core.display.Math object>

rank(A) = 2
A.inv() raises: NonInvertibleMatrixError — Matrix det == 0; not invertible.


## 4. 幾何意義：體積塌縮

$\lvert\det(A)\rvert$ 是以各行（或列）向量為邊的平行四邊形／平行多面體的**體積**。

- 非奇異：$\det \neq 0$ → 有正體積 → 線性變換把空間「攤開」
- 奇異：$\det = 0$ → 體積為 **0** → 向量共線／共面，線性變換把空間**壓扁**到較低維子空間

例如 $S = \begin{pmatrix} 1 & 2 \\ 2 & 4 \end{pmatrix}$ 的兩行互為倍數，落在同一條線上，平行四邊形面積為 0。

In [5]:
# Area interpretation for 2x2
print("S: rows are parallel => area = |det| =")
display_large(sp.Abs(S.det()))

print("N: rows independent => area = |det| =")
display_large(sp.Abs(N.det()))

# Column vectors of S
print("Column vectors of S:")
display_large(S.col(0))
display_large(S.col(1))
print("col1 == 2 * col0?", S.col(1) == 2 * S.col(0))

S: rows are parallel => area = |det| =


<IPython.core.display.Math object>

N: rows independent => area = |det| =


<IPython.core.display.Math object>

Column vectors of S:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

col1 == 2 * col0? True


## 5. 齊次方程組 $A\mathbf{x} = \mathbf{0}$

對任意 $A$，零向量 $\mathbf{x}=\mathbf{0}$ 永遠是解（**平凡解**）。

| $A$ | $\det(A)$ | $A\mathbf{x}=\mathbf{0}$ |
|-----|-----------|-------------------------|
| 非奇異 | $\neq 0$ | **只有**平凡解 $\mathbf{x}=\mathbf{0}$ |
| 奇異 | $= 0$ | 有**無限多**非平凡解（零空間維度 $= n - \operatorname{rank}(A)$） |

這正是「奇異 $\Leftrightarrow$ 行／列線性相關」的另一面：存在非零 $\mathbf{x}$ 使 $A\mathbf{x}=\mathbf{0}$。

In [6]:
# Nullspace of singular S
print("S =")
display_huge(S)

null_S = S.nullspace()
print(f"dim(nullspace) = {len(null_S)}  (= n - rank = {S.cols} - {S.rank()})")
print("Basis vector of nullspace:")
display_huge(null_S[0])

# Verify S * x = 0
x0 = null_S[0]
print("S * x0 =")
display_large(S * x0)

# Nonsingular N: only trivial nullspace
print("\nN.nullspace() =", N.nullspace())
print("Only the zero vector => unique trivial solution")

S =


<IPython.core.display.Math object>

dim(nullspace) = 1  (= n - rank = 2 - 1)
Basis vector of nullspace:


<IPython.core.display.Math object>

S * x0 =


<IPython.core.display.Math object>


N.nullspace() = []
Only the zero vector => unique trivial solution


### 5.1 用參數寫出全部解

對 $S\mathbf{x}=\mathbf{0}$：

$$
\begin{pmatrix} 1 & 2 \\ 2 & 4 \end{pmatrix}
\begin{pmatrix} x \\ y \end{pmatrix}
= \begin{pmatrix} 0 \\ 0 \end{pmatrix}
\quad\Rightarrow\quad
x + 2y = 0
\quad\Rightarrow\quad
\begin{pmatrix} x \\ y \end{pmatrix}
= t \begin{pmatrix} -2 \\ 1 \end{pmatrix},\; t \in \mathbb{R}
$$

In [7]:
t = sp.symbols("t")
general = t * null_S[0]
print("General solution of S x = 0:")
display_huge(general)

for val in [0, 1, -3]:
    xv = general.subs(t, val)
    print(f"t = {val}: x = {xv.T},  S x = {(S * xv).T}")

General solution of S x = 0:


<IPython.core.display.Math object>

t = 0: x = Matrix([[0, 0]]),  S x = Matrix([[0, 0]])
t = 1: x = Matrix([[-2, 1]]),  S x = Matrix([[0, 0]])
t = -3: x = Matrix([[6, -3]]),  S x = Matrix([[0, 0]])


## 6. 非齊次方程組 $A\mathbf{x} = \mathbf{b}$（$A$ 奇異時）

當 $A$ 奇異，$A\mathbf{x}=\mathbf{b}$ **不可能有唯一解**，只剩兩種情形：

| 條件 | 結果 |
|------|------|
| $\operatorname{rank}(A) < \operatorname{rank}([A\mid b])$ | **無解** |
| $\operatorname{rank}(A) = \operatorname{rank}([A\mid b]) < n$ | **無限多解** |

（回顧第 003 章：用增廣矩陣的 RREF 判斷。）

### 6.1 無限多解

$$
\begin{cases}
x + 2y = 5 \\
2x + 4y = 10
\end{cases}
\quad\text{（第二式 = 第一式的 2 倍，一致）}
$$

In [8]:
A_inf = sp.Matrix([[1, 2], [2, 4]])
b_inf = sp.Matrix([5, 10])
aug_inf = A_inf.row_join(b_inf)

print("Augmented [A|b] =")
display_huge(aug_inf)

R_inf, pivots_inf = aug_inf.rref()
print("RREF =")
display_huge(R_inf)

print(f"rank(A) = {A_inf.rank()}, rank([A|b]) = {aug_inf.rank()}, n = {A_inf.cols}")
print("Consistent & rank < n  =>  infinitely many solutions")

# Parametric: x + 2y = 5  =>  x = 5 - 2t, y = t
sol_inf = sp.Matrix([5 - 2 * t, t])
print("Parametric solution (y = t):")
display_huge(sol_inf)

for val in [0, 1, 2]:
    xv = sol_inf.subs(t, val)
    print(f"t = {val}: A x = {A_inf * xv}, b = {b_inf}, match? {A_inf * xv == b_inf}")

Augmented [A|b] =


<IPython.core.display.Math object>

RREF =


<IPython.core.display.Math object>

rank(A) = 1, rank([A|b]) = 1, n = 2
Consistent & rank < n  =>  infinitely many solutions
Parametric solution (y = t):


<IPython.core.display.Math object>

t = 0: A x = Matrix([[5], [10]]), b = Matrix([[5], [10]]), match? True
t = 1: A x = Matrix([[5], [10]]), b = Matrix([[5], [10]]), match? True
t = 2: A x = Matrix([[5], [10]]), b = Matrix([[5], [10]]), match? True


### 6.2 無解

$$
\begin{cases}
x + 2y = 5 \\
2x + 4y = 11
\end{cases}
\quad\text{（左端成比例，右端不成比例 → 矛盾）}
$$

In [9]:
A_none = sp.Matrix([[1, 2], [2, 4]])
b_none = sp.Matrix([5, 11])
aug_none = A_none.row_join(b_none)

print("Augmented [A|b] =")
display_huge(aug_none)

R_none, pivots_none = aug_none.rref()
print("RREF =")
display_huge(R_none)

print(f"rank(A) = {A_none.rank()}, rank([A|b]) = {aug_none.rank()}")
print("Inconsistent: rank(A) < rank([A|b])  =>  no solution")

try:
    A_none.solve(b_none)
except Exception as e:
    print("A.solve(b) raises:", type(e).__name__, "—", e)

Augmented [A|b] =


<IPython.core.display.Math object>

RREF =


<IPython.core.display.Math object>

rank(A) = 1, rank([A|b]) = 2
Inconsistent: rank(A) < rank([A|b])  =>  no solution
A.solve(b) raises: NonInvertibleMatrixError — Matrix det == 0; not invertible.


## 7. 綜合練習

考慮

$$
B = \begin{pmatrix}
1 & 1 & 1 \\
2 & 3 & 4 \\
3 & 4 & 5
\end{pmatrix}
$$

請判斷 $B$ 是否奇異，並：

1. 計算 $\det(B)$、秩、RREF
2. 找出 $B\mathbf{x}=\mathbf{0}$ 的非平凡解（若存在）
3. 對 $\mathbf{b}=(1,2,3)^T$，判斷 $B\mathbf{x}=\mathbf{b}$ 有解與否

In [10]:
B = sp.Matrix([
    [1, 1, 1],
    [2, 3, 4],
    [3, 4, 5],
])

print("B =")
display_huge(B)

print("det(B) =")
display_large(B.det())
print(f"rank(B) = {B.rank()}, n = {B.rows}")
print("Singular?", B.det() == 0)

R_B, pivots_B = B.rref()
print("RREF =")
display_huge(R_B)
print("Pivots:", list(pivots_B))

# Nullspace
null_B = B.nullspace()
print(f"\nnullspace dimension = {len(null_B)}")
if null_B:
    print("Nontrivial solution basis:")
    display_huge(null_B[0])
    print("B * x0 =")
    display_large(B * null_B[0])

# Bx = b with b = (1, 2, 3)^T
b = sp.Matrix([1, 2, 3])
aug_B = B.row_join(b)
print("\nAugmented [B|b] RREF =")
display_huge(aug_B.rref()[0])
print(f"rank(B) = {B.rank()}, rank([B|b]) = {aug_B.rank()}")
if B.rank() < aug_B.rank():
    print("No solution")
elif B.rank() < B.cols:
    print("Infinitely many solutions")
else:
    print("Unique solution")

B =


<IPython.core.display.Math object>

det(B) =


<IPython.core.display.Math object>

rank(B) = 2, n = 3
Singular? True
RREF =


...

B * x0 =


<IPython.core.display.Math object>


Augmented [B|b] RREF =


<IPython.core.display.Math object>

rank(B) = 2, rank([B|b]) = 2
Infinitely many solutions


## 小結

| 概念 | SymPy | 重點 |
|------|-------|------|
| 奇異判定 | `A.det() == 0` | 方陣不可逆 $\Leftrightarrow$ $\det=0$ |
| 秩判定 | `A.rank() < n` | 等價於奇異 |
| RREF | `A.rref()` | 奇異 $\Leftrightarrow$ RREF $\neq I$ |
| 反矩陣 | `A.inv()` | 奇異時拋出 `NonInvertibleMatrixError` |
| 零空間 | `A.nullspace()` | 奇異 $\Leftrightarrow$ 有非平凡核向量 |
| $A\mathbf{x}=\mathbf{b}$ | 比較 $\operatorname{rank}(A)$ 與 $\operatorname{rank}([A\mid b])$ | 無解或無限多解，絕無唯一解 |

**記住：** 奇異 = 行列式為零 = 不可逆 = 不滿秩 = 列／行線性相關 = 零空間有非零向量。這些是同一件事的不同說法。